# State + Memory + HITL with LangGraph + Milvus for Telecom Domain
## Beginner-Friendly Step-by-Step Hands-On Lab

This notebook explains and demonstrates:

- **State object**
- **Reducers**
- **Short-term vs Long-term memory**
- **HITL (Human in the Loop)**
- **Approval workflow**
- **Adding memory**
- **Adding a HITL node**
- **Milvus vector store usage in a telecom workflow**

---

## What you will build

A small telecom workflow for a case like:

> "My mobile internet is slow in Pune and this customer has complained multiple times."

The workflow will:

1. read customer and incident details
2. retrieve guidance from Milvus
3. build a diagnosis
4. pause for human approval if escalation is needed
5. continue after approval
6. store memory about the decision

---

## Beginner goal

By the end of this notebook, you should understand:

- what **state** means in LangGraph
- what a **reducer** does
- what **short-term memory** and **long-term memory** mean
- what **HITL** means in a real workflow
- how to model a telecom approval flow

## Why LangGraph?

LangGraph is built around **state, nodes, and edges**. It is useful when workflows are not just one straight line and when you need:
- branching
- memory
- approval steps
- resumable logic

## Why Milvus?

Milvus is a vector database. It helps retrieve relevant telecom SOP or RCA guidance using semantic search plus filters like:
- city
- issue type
- severity

This makes it useful for telecom support, incident resolution, and RAG workflows.

In [ ]:
# Uncomment if needed in a fresh environment
# %pip install -U langgraph pymilvus pandas

## Step 1 — Imports

In [ ]:
from __future__ import annotations

from typing import TypedDict, Dict, Any, List, Optional
from pprint import pprint
from copy import deepcopy
import pandas as pd

from langgraph.graph import StateGraph, END

from pymilvus import (
    connections,
    utility,
    Collection,
    FieldSchema,
    CollectionSchema,
    DataType,
)

## Step 2 — Concept: What is a State object?

A **state object** is the shared data that moves through the graph.

Think of it like a container that holds:
- customer info
- issue type
- retrieved documents
- diagnosis
- approval status
- memory notes

Each node can:
- read from state
- add new data to state
- update existing data in state

So state is the common memory of the workflow while it runs.

## Step 3 — Concept: What is a reducer?

A **reducer** is the rule used to combine updates.

Beginner idea:
- if two nodes both update the same field, how should those updates be combined?

Examples:
- append to a list
- merge dictionaries
- overwrite a field

In this notebook, we use a simple list-style reducer idea for `audit_log`:
- every step can add one or more log messages
- the final audit log is the combined list

In [ ]:
def append_list_reducer(old_list, new_list):
    old_list = old_list or []
    new_list = new_list or []
    return old_list + new_list

append_list_reducer(["step1"], ["step2", "step3"])

## Step 4 — Concept: Short-term vs Long-term memory

### Short-term memory
Short-term memory is memory for the **current workflow run**.

Example:
- current customer
- current diagnosis
- approval needed or not
- retrieved guidance
- audit trail

This usually lives inside the LangGraph state.

### Long-term memory
Long-term memory is memory that should survive beyond one run.

Example:
- "Customer CUST_1001 has repeated slow internet complaints"
- "Pune slow internet often maps to congestion during peak hours"
- "This customer was previously escalated to NOC"

In this notebook:
- **short-term memory** lives in the graph state
- **long-term memory** is stored in a simple in-memory store after approval

## Step 5 — Concept: What is HITL?

**HITL = Human in the Loop**

It means the workflow does not always decide everything automatically.

At some point, the workflow may say:

> "This needs human approval before continuing."

Example telecom cases:
- high-priority escalation
- repeated complaint from premium customer
- compensation decision
- NOC escalation approval

So HITL is useful when:
- business risk is high
- human judgment is required
- automatic action should be controlled

## Step 6 — Telecom guidance documents for Milvus

We create a small telecom knowledge base for:
- slow internet
- call drop
- packet loss
- repeated complaint handling

In [ ]:
telecom_docs = [
    {
        "doc_id": "DOC001",
        "city": "Pune",
        "issue_type": "slow_internet",
        "content": "If tower utilization exceeds 90 percent during peak hours in Pune, classify the issue as likely congestion and notify NOC."
    },
    {
        "doc_id": "DOC002",
        "city": "Pune",
        "issue_type": "slow_internet",
        "content": "For repeated slow internet complaints in Pune, review complaint history, check outage status, and consider escalation to network operations."
    },
    {
        "doc_id": "DOC003",
        "city": "Mumbai",
        "issue_type": "packet_loss",
        "content": "If packet loss affects browsing in Mumbai, investigate transport path degradation and escalate to network operations."
    },
    {
        "doc_id": "DOC004",
        "city": "Pune",
        "issue_type": "call_drop",
        "content": "For frequent call drops in Pune, verify radio quality, SINR, and antenna alignment."
    },
]

pd.DataFrame(telecom_docs)

## Step 7 — Create demo embeddings

We use deterministic demo vectors so the notebook stays simple.

In production, replace this with a real embedding model.

In [ ]:
DIM = 10

def demo_embedding(text: str, dim: int = DIM) -> List[float]:
    vals = [0.0] * dim
    for i, ch in enumerate(text.lower()):
        vals[i % dim] += (ord(ch) % 29) / 100.0
    total = sum(vals) or 1.0
    return [round(v / total, 6) for v in vals]

for row in telecom_docs:
    row["embedding"] = demo_embedding(row["content"])

telecom_docs[0]["embedding"][:5]

## Step 8 — Connect to Milvus

In [ ]:
MILVUS_HOST = "localhost"
MILVUS_PORT = "19530"
COLLECTION_NAME = "telecom_state_memory_hitl"

connections.connect(alias="default", host=MILVUS_HOST, port=MILVUS_PORT)
print("Connected to Milvus")

## Step 9 — Create Milvus collection

In [ ]:
if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)
    print(f"Dropped existing collection: {COLLECTION_NAME}")

fields = [
    FieldSchema(name="pk", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="doc_id", dtype=DataType.VARCHAR, max_length=32),
    FieldSchema(name="city", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="issue_type", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="content", dtype=DataType.VARCHAR, max_length=2048),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=DIM),
]

schema = CollectionSchema(fields=fields, description="Telecom docs for State+Memory+HITL lab")
collection = Collection(name=COLLECTION_NAME, schema=schema)

index_params = {
    "index_type": "IVF_FLAT",
    "metric_type": "L2",
    "params": {"nlist": 32},
}
collection.create_index(field_name="embedding", index_params=index_params)

print("Collection created")

## Step 10 — Insert documents into Milvus

In [ ]:
collection.insert(telecom_docs)
collection.flush()
collection.load()

print("Collection loaded")
print("Entities:", collection.num_entities)

## Step 11 — Helper functions

We now define:
- customer data
- outage data
- guidance retrieval
- long-term memory store

In [ ]:
CUSTOMERS = {
    "CUST_1001": {
        "customer_id": "CUST_1001",
        "name": "Ravi Patil",
        "city": "Pune",
        "plan": "Premium 5G",
        "network_type": "5G",
        "recent_complaints": 3,
    },
    "CUST_2001": {
        "customer_id": "CUST_2001",
        "name": "Meera Shah",
        "city": "Mumbai",
        "plan": "Standard 4G",
        "network_type": "4G",
        "recent_complaints": 0,
    },
}

OUTAGES = [
    {"city": "Pune", "issue_type": "slow_internet", "outage_detected": False, "summary": "No citywide outage found."},
    {"city": "Mumbai", "issue_type": "packet_loss", "outage_detected": True, "summary": "Packet loss incident active in Mumbai region."},
]

LONG_TERM_MEMORY = {}

In [ ]:
def get_customer_profile(customer_id: str) -> Dict[str, Any]:
    return CUSTOMERS[customer_id]

def check_outage(city: str, issue_type: str) -> Dict[str, Any]:
    match = next(
        (row for row in OUTAGES if row["city"] == city and row["issue_type"] == issue_type),
        {"city": city, "issue_type": issue_type, "outage_detected": False, "summary": "No outage found."}
    )
    return match

def retrieve_guidance(query_text: str, city_filter: str | None = None, issue_filter: str | None = None, top_k: int = 2):
    query_vec = demo_embedding(query_text)

    filters = []
    if city_filter:
        filters.append(f'city == "{city_filter}"')
    if issue_filter:
        filters.append(f'issue_type == "{issue_filter}"')
    expr = " and ".join(filters) if filters else None

    results = collection.search(
        data=[query_vec],
        anns_field="embedding",
        param={"metric_type": "L2", "params": {"nprobe": 10}},
        limit=top_k,
        expr=expr,
        output_fields=["doc_id", "city", "issue_type", "content"],
    )

    rows = []
    for hits in results:
        for hit in hits:
            rows.append({
                "doc_id": hit.entity.get("doc_id"),
                "city": hit.entity.get("city"),
                "issue_type": hit.entity.get("issue_type"),
                "content": hit.entity.get("content"),
                "distance": float(hit.distance),
            })
    return rows

## Step 12 — Define the workflow state

This state includes:

- business input
- retrieved guidance
- diagnosis
- approval status
- audit log
- short-term memory notes
- long-term memory save flag

In [ ]:
class TelecomApprovalState(TypedDict, total=False):
    customer_id: str
    issue_type: str
    user_query: str

    customer_profile: Dict[str, Any]
    outage_info: Dict[str, Any]
    guidance_results: List[Dict[str, Any]]

    diagnosis: str
    next_action: str

    approval_required: bool
    approved: bool
    approval_comment: str

    audit_log: List[str]
    short_term_memory: List[str]
    memory_saved: bool

    final_response: str

## Step 13 — Define nodes

We will create nodes for:
- get profile
- check outage
- retrieve guidance
- diagnose
- decide approval requirement
- human approval node
- save memory
- final response

We also maintain:
- `audit_log`
- `short_term_memory`

In [ ]:
def get_profile_node(state: TelecomApprovalState) -> Dict[str, Any]:
    profile = get_customer_profile(state["customer_id"])
    return {
        "customer_profile": profile,
        "audit_log": append_list_reducer(state.get("audit_log"), ["Profile loaded"]),
        "short_term_memory": append_list_reducer(state.get("short_term_memory"), [f"Customer city is {profile['city']}"]),
    }

def check_outage_node(state: TelecomApprovalState) -> Dict[str, Any]:
    city = state["customer_profile"]["city"]
    issue_type = state["issue_type"]
    outage = check_outage(city, issue_type)
    return {
        "outage_info": outage,
        "audit_log": append_list_reducer(state.get("audit_log"), [f"Outage checked: {outage['summary']}"]),
        "short_term_memory": append_list_reducer(state.get("short_term_memory"), [f"Outage status = {outage['outage_detected']}"]),
    }

def retrieve_guidance_node(state: TelecomApprovalState) -> Dict[str, Any]:
    city = state["customer_profile"]["city"]
    issue_type = state["issue_type"]
    rows = retrieve_guidance(
        state["user_query"],
        city_filter=city,
        issue_filter=issue_type,
        top_k=2,
    )
    return {
        "guidance_results": rows,
        "audit_log": append_list_reducer(state.get("audit_log"), [f"Retrieved {len(rows)} guidance docs from Milvus"]),
    }

def diagnose_node(state: TelecomApprovalState) -> Dict[str, Any]:
    if state["outage_info"]["outage_detected"]:
        diagnosis = f"Confirmed outage: {state['outage_info']['summary']}"
        next_action = "Communicate outage and escalate to NOC."
    else:
        guidance = state.get("guidance_results", [])
        if guidance:
            diagnosis = guidance[0]["content"]
            next_action = "Follow guidance and consider escalation if complaints are repeated."
        else:
            diagnosis = "No matching guidance found."
            next_action = "Manual triage needed."

    return {
        "diagnosis": diagnosis,
        "next_action": next_action,
        "audit_log": append_list_reducer(state.get("audit_log"), ["Diagnosis created"]),
        "short_term_memory": append_list_reducer(state.get("short_term_memory"), [f"Diagnosis prepared: {diagnosis}"]),
    }

def decide_approval_node(state: TelecomApprovalState) -> Dict[str, Any]:
    profile = state["customer_profile"]
    approval_required = False

    # Beginner-friendly business rule:
    # repeated complaint or premium plan + escalation-like next action => approval
    if profile["recent_complaints"] >= 2 or "escalate" in state["next_action"].lower():
        approval_required = True

    return {
        "approval_required": approval_required,
        "audit_log": append_list_reducer(state.get("audit_log"), [f"Approval required = {approval_required}"]),
    }

## Step 14 — HITL node

This node simulates human approval.

In a real system:
- this may pause and wait for a supervisor
- or use a human task queue
- or call a UI approval action

For this beginner lab, we simulate the human decision with simple rules:
- if premium customer and repeated complaint -> approve
- otherwise approve by default

In [ ]:
def hitl_approval_node(state: TelecomApprovalState) -> Dict[str, Any]:
    if not state.get("approval_required", False):
        return {
            "approved": True,
            "approval_comment": "Approval not required.",
            "audit_log": append_list_reducer(state.get("audit_log"), ["Skipped HITL approval"]),
        }

    profile = state["customer_profile"]

    # Simulated human decision
    if profile["plan"] == "Premium 5G" and profile["recent_complaints"] >= 2:
        approved = True
        comment = "Approved by supervisor due to repeated complaint and premium customer status."
    else:
        approved = True
        comment = "Approved after review."

    return {
        "approved": approved,
        "approval_comment": comment,
        "audit_log": append_list_reducer(state.get("audit_log"), [f"HITL decision: {comment}"]),
        "short_term_memory": append_list_reducer(state.get("short_term_memory"), [f"Approval outcome: {comment}"]),
    }

## Step 15 — Save long-term memory

This node writes a small memory record into a simple in-memory store.

In a real system, long-term memory may be:
- a database
- a profile store
- a vector DB
- a conversation memory system

In [ ]:
def save_memory_node(state: TelecomApprovalState) -> Dict[str, Any]:
    customer_id = state["customer_id"]

    LONG_TERM_MEMORY[customer_id] = {
        "latest_issue_type": state["issue_type"],
        "latest_diagnosis": state["diagnosis"],
        "approval_required": state.get("approval_required", False),
        "approved": state.get("approved", False),
        "approval_comment": state.get("approval_comment", ""),
        "recent_complaint_count": state["customer_profile"]["recent_complaints"],
    }

    return {
        "memory_saved": True,
        "audit_log": append_list_reducer(state.get("audit_log"), ["Long-term memory saved"]),
    }

## Step 16 — Final response node

In [ ]:
def final_response_node(state: TelecomApprovalState) -> Dict[str, Any]:
    profile = state["customer_profile"]

    response = (
        f"Customer: {profile['name']}\n"
        f"City: {profile['city']}\n"
        f"Issue Type: {state['issue_type']}\n"
        f"Diagnosis: {state['diagnosis']}\n"
        f"Next Action: {state['next_action']}\n"
        f"Approval Required: {state.get('approval_required')}\n"
        f"Approved: {state.get('approved')}\n"
        f"Approval Comment: {state.get('approval_comment')}\n"
        f"Memory Saved: {state.get('memory_saved')}"
    )

    return {
        "final_response": response,
        "audit_log": append_list_reducer(state.get("audit_log"), ["Final response created"]),
    }

## Step 17 — Conditional routing

Now we define how the graph should route.

Rule:
- after deciding approval, always go to the HITL node
- after HITL, save memory
- then create final response

This makes the workflow explicit and easy to understand.

In [ ]:
def route_after_approval_decision(state: TelecomApprovalState) -> str:
    return "hitl_approval"

## Step 18 — Build the approval workflow graph

In [ ]:
builder = StateGraph(TelecomApprovalState)

builder.add_node("get_profile", get_profile_node)
builder.add_node("check_outage", check_outage_node)
builder.add_node("retrieve_guidance", retrieve_guidance_node)
builder.add_node("diagnose", diagnose_node)
builder.add_node("decide_approval", decide_approval_node)
builder.add_node("hitl_approval", hitl_approval_node)
builder.add_node("save_memory", save_memory_node)
builder.add_node("final_response", final_response_node)

builder.set_entry_point("get_profile")
builder.add_edge("get_profile", "check_outage")
builder.add_edge("check_outage", "retrieve_guidance")
builder.add_edge("retrieve_guidance", "diagnose")
builder.add_edge("diagnose", "decide_approval")

builder.add_conditional_edges(
    "decide_approval",
    route_after_approval_decision,
    {
        "hitl_approval": "hitl_approval"
    }
)

builder.add_edge("hitl_approval", "save_memory")
builder.add_edge("save_memory", "final_response")
builder.add_edge("final_response", END)

approval_graph = builder.compile()
print("Approval workflow graph compiled")

## Step 19 — Run the workflow for a repeated complaint in Pune

Expected flow:
- profile loaded
- outage checked
- guidance retrieved
- diagnosis created
- approval required
- HITL approval simulated
- memory saved
- final response produced

In [ ]:
state_1 = {
    "customer_id": "CUST_1001",
    "issue_type": "slow_internet",
    "user_query": "My mobile internet is slow in Pune and this has happened repeatedly.",
    "audit_log": [],
    "short_term_memory": [],
}

result_1 = approval_graph.invoke(state_1)
print(result_1["final_response"])

## Step 20 — Inspect short-term memory

This is the memory that lived inside the workflow state.

In [ ]:
result_1["short_term_memory"]

## Step 21 — Inspect audit log

This shows how reducers helped us keep a combined list of workflow events.

In [ ]:
result_1["audit_log"]

## Step 22 — Inspect long-term memory

This is the memory that survives outside the current graph run.

In [ ]:
LONG_TERM_MEMORY

## Step 23 — Run a second case

Now test a different customer.

In [ ]:
state_2 = {
    "customer_id": "CUST_2001",
    "issue_type": "packet_loss",
    "user_query": "There is packet loss affecting browsing in Mumbai.",
    "audit_log": [],
    "short_term_memory": [],
}

result_2 = approval_graph.invoke(state_2)
print(result_2["final_response"])

## Step 24 — Approval workflow explained step by step

### Flow
```text
Get Profile
   ↓
Check Outage
   ↓
Retrieve Guidance from Milvus
   ↓
Diagnose
   ↓
Decide Approval Needed
   ↓
HITL Approval Node
   ↓
Save Long-Term Memory
   ↓
Final Response
```

### What was short-term memory?
- state fields
- audit log
- short_term_memory notes

### What was long-term memory?
- saved customer memory in `LONG_TERM_MEMORY`

## Step 25 — Key beginner lessons

### State object
The shared container of workflow data.

### Reducers
A way to combine repeated updates, especially useful for logs and memory notes.

### Short-term memory
Memory inside the current workflow state.

### Long-term memory
Memory stored outside the current run so future workflows can reuse it.

### HITL
A human approval step that helps control risky or important actions.

### Milvus role
Milvus retrieves telecom guidance that helps the workflow produce better decisions.

## Exercises

1. Add a rejection path in the HITL node
2. Save more fields into long-term memory
3. Add ticket creation after approval
4. Add another issue type such as call_drop
5. Add a second reducer for a separate escalation log

# Summary

This notebook demonstrated:
- State object
- Reducers
- Short-term memory
- Long-term memory
- HITL
- Approval workflow
- Memory saving
- HITL node
- Milvus-backed telecom guidance retrieval

This gives you a strong beginner foundation for State + Memory + HITL in LangGraph.